<a href="https://colab.research.google.com/github/vanashri-18/CSA6101-Digital-Forensics-and-Cybercrime-Investigation/blob/main/Windows_Event_Log_Failed_Login_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Aim**

To write a Python program to analyze Windows Event Log data and identify repeated failed login attempts by detecting multiple failed authentication events within the log.

**Algorithm**

Import the required Python libraries.

Create/load Windows Event Log data in CSV format.

Read the event log using Pandas.

Filter events with Event ID 4625, which represents a failed Windows logon.

Group the failed login attempts based on username and source IP address.

Count the number of failed attempts for each user/IP combination.

Set a threshold for repeated failed login attempts.

Display accounts/IP addresses whose failed attempts exceed the threshold.

Generate a summary of suspicious login activity.

In [1]:
# ============================================
# Windows Event Log - Failed Login Analysis
# ============================================

import pandas as pd
from io import StringIO

# --------------------------------------------
# Step 1: Create sample Windows Event Log data
# --------------------------------------------

log_data = """DateTime,EventID,Username,SourceIP,LogonType
2026-08-18 09:01:10,4625,admin,192.168.1.50,3
2026-08-18 09:02:15,4625,admin,192.168.1.50,3
2026-08-18 09:03:20,4625,admin,192.168.1.50,3
2026-08-18 09:04:25,4625,admin,192.168.1.50,3
2026-08-18 09:05:30,4625,admin,192.168.1.50,3
2026-08-18 09:06:35,4624,admin,192.168.1.50,3
2026-08-18 09:10:10,4625,user1,192.168.1.20,2
2026-08-18 09:11:15,4625,user1,192.168.1.20,2
2026-08-18 09:12:20,4625,user1,192.168.1.20,2
2026-08-18 09:20:10,4625,user2,192.168.1.30,3
2026-08-18 09:25:15,4625,guest,10.0.0.15,3
2026-08-18 09:26:20,4625,guest,10.0.0.15,3
2026-08-18 09:27:25,4625,guest,10.0.0.15,3
2026-08-18 09:28:30,4625,guest,10.0.0.15,3
"""

# Convert the sample data into a DataFrame
df = pd.read_csv(StringIO(log_data))

# --------------------------------------------
# Step 2: Display original event log
# --------------------------------------------

print("========== WINDOWS EVENT LOG ==========\n")
print(df.to_string(index=False))

# --------------------------------------------
# Step 3: Filter failed login attempts
# Event ID 4625 = Failed Logon
# --------------------------------------------

failed_logins = df[df["EventID"] == 4625].copy()

print("\n\n========== FAILED LOGIN ATTEMPTS ==========\n")
print(failed_logins.to_string(index=False))

# --------------------------------------------
# Step 4: Count failed attempts
# Group by Username and Source IP
# --------------------------------------------

login_counts = (
    failed_logins
    .groupby(["Username", "SourceIP"])
    .size()
    .reset_index(name="Failed_Attempts")
)

print("\n\n========== LOGIN ATTEMPT SUMMARY ==========\n")
print(login_counts.to_string(index=False))

# --------------------------------------------
# Step 5: Define detection threshold
# --------------------------------------------

THRESHOLD = 3

# Identify repeated failed login attempts
suspicious_logins = login_counts[
    login_counts["Failed_Attempts"] >= THRESHOLD
].copy()

# --------------------------------------------
# Step 6: Display suspicious activity
# --------------------------------------------

print("\n\n========== SUSPICIOUS LOGIN ACTIVITY ==========\n")

if suspicious_logins.empty:
    print("No repeated failed login attempts detected.")
else:
    print(
        suspicious_logins.to_string(index=False)
    )

# --------------------------------------------
# Step 7: Generate security alert
# --------------------------------------------

print("\n\n========== SECURITY ALERTS ==========\n")

for _, row in suspicious_logins.iterrows():
    print(
        f"[ALERT] User '{row['Username']}' from IP "
        f"{row['SourceIP']} had {row['Failed_Attempts']} "
        f"failed login attempts."
    )

# --------------------------------------------
# Step 8: Overall statistics
# --------------------------------------------

print("\n\n========== ANALYSIS SUMMARY ==========\n")

print("Total Event Log Entries :", len(df))
print("Total Failed Logins     :", len(failed_logins))
print("Unique Users with Failed Login :",
      failed_logins["Username"].nunique())
print("Suspicious User/IP Pairs :", len(suspicious_logins))

print("\nAnalysis completed successfully.")

========== WINDOWS EVENT LOG ==========

           DateTime  EventID Username     SourceIP  LogonType
2026-08-18 09:01:10     4625    admin 192.168.1.50          3
2026-08-18 09:02:15     4625    admin 192.168.1.50          3
2026-08-18 09:03:20     4625    admin 192.168.1.50          3
2026-08-18 09:04:25     4625    admin 192.168.1.50          3
2026-08-18 09:05:30     4625    admin 192.168.1.50          3
2026-08-18 09:06:35     4624    admin 192.168.1.50          3
2026-08-18 09:10:10     4625    user1 192.168.1.20          2
2026-08-18 09:11:15     4625    user1 192.168.1.20          2
2026-08-18 09:12:20     4625    user1 192.168.1.20          2
2026-08-18 09:20:10     4625    user2 192.168.1.30          3
2026-08-18 09:25:15     4625    guest    10.0.0.15          3
2026-08-18 09:26:20     4625    guest    10.0.0.15          3
2026-08-18 09:27:25     4625    guest    10.0.0.15          3
2026-08-18 09:28:30     4625    guest    10.0.0.15          3


========== FAILED LOGIN ATT

**Result**

The Python program successfully analyzed Windows Event Log data and detected repeated failed login attempts using Event ID 4625. Users and source IP addresses exceeding the threshold of 3 failed attempts were identified as suspicious, helping security analysts detect possible brute-force or password-guessing attacks.